# Predict — long-form, diarization, code-switch, APIs (M4–M7)

Runs the **fine-tuned model** through the full stack and fills every report. One
prediction path (`run_over_manifest`) for our model, the long-form pipeline, and the
commercial APIs; DER has its own scorer. Thin driver over the tested `shono` package.

Prerequisites: `prepare_data.ipynb` (manifests) and `train_whisper_medium.ipynb`
(the fine-tuned checkpoint) have run.

In [ ]:
!pip install -q 'transformers>=4.44' 'faster-whisper==1.2.1' 'ctranslate2==4.4.0' \
    'pyannote.audio>=4.0' librosa soundfile 'google-cloud-speech>=2.27' deepgram-sdk
!pip install -q -e /kaggle/working/shono

In [ ]:
import os
from pathlib import Path

os.environ.setdefault('HF_TOKEN', '')  # gated pyannote + Hub (Kaggle secrets)
REPO = Path('/kaggle/working/shono'); DATA = REPO / 'data'
PREDS = REPO / 'reports' / 'predictions'; PREDS.mkdir(parents=True, exist_ok=True)
AUDIO_ROOT = Path('/kaggle/input')
FT_MODEL = '/kaggle/working/whisper-medium-bn-ft'   # the M3 checkpoint
CT2_MODEL = '/kaggle/working/whisper-medium-bn-ct2'  # CTranslate2 conversion
OUR_SYSTEM = 'shono whisper-medium (fine-tuned)'; OUR_SLUG = 'shono-medium'

## M4 — long-form: CT2-convert → pipeline → predictions + RTF

The recording-level eval collapses the segment manifest (score the whole recording
against the merged transcript — the honest long-form protocol). RTF is measured by
the transcribe CLI on one file.

In [ ]:
from shono.api import run_over_manifest
from shono.data import Manifest, collapse_to_recordings
from shono.provenance import RunContext
from shono.transcribe import (
    FasterWhisperTranscriber, LongFormPipelineTranscriber, LongFormTranscriber, SileroVAD,
)
from shono.transcribe.convert import to_ct2

to_ct2(FT_MODEL, CT2_MODEL, quantization='float16')

loop_seg = Manifest.from_jsonl(DATA / 'bengali-loop-sample.manifest.jsonl')
loop_rec = collapse_to_recordings(loop_seg)
loop_rec.to_jsonl(DATA / 'bengali-loop-test.manifest.jsonl')  # recording-level, for the report

pipeline = LongFormTranscriber(SileroVAD(), FasterWhisperTranscriber(CT2_MODEL, language='bn'))
adapter = LongFormPipelineTranscriber(pipeline)
ctx = RunContext.capture(0, {'model': FT_MODEL}, command='predict.ipynb', repo=str(REPO))
preds = run_over_manifest(adapter, loop_rec, OUR_SYSTEM, audio_root=AUDIO_ROOT, run_context=ctx)
preds.to_jsonl(PREDS / f'{OUR_SLUG}__bengali-loop-test.jsonl')
print(f'long-form: {len(preds.hypotheses)} recordings transcribed')

In [ ]:
# Real-time factor on one long recording (S3: RTF < 1.0 on a T4).
sample = next((loop_rec.segments[0].audio,))  # any recording file
!cd /kaggle/working/shono && python -m shono.transcribe {AUDIO_ROOT}/{sample} --model {CT2_MODEL}
!cd /kaggle/working/shono && python -m shono.eval --report longform

## M5 — diarization: pyannote + VAD-intersection → DER

Diarize each recording, intersect with the VAD speech regions, and score against the
reference RTTMs with the corpus DER + bootstrap CI under the stated collar/overlap
protocol.

In [ ]:
from shono.diarize import (
    DERConfig, PyannoteDiarizer, corpus_der, der_bootstrap_ci, load_rttm,
    render_der_report, vad_intersection,
)

DIAR_ROOT = Path('/kaggle/input/bengali-loop/diarization')  # audio + reference RTTMs
recordings = [p.stem for p in sorted(DIAR_ROOT.glob('*.rttm'))]
diarizer = PyannoteDiarizer(hf_token=os.environ['HF_TOKEN'])
vad = SileroVAD()

pairs = []
for rec in recordings:
    audio = str(DIAR_ROOT / f'{rec}.wav')
    hyp = vad_intersection(diarizer.diarize(audio), vad.detect(audio))
    ref = load_rttm(DIAR_ROOT / f'{rec}.rttm', file_id=rec)
    pairs.append((ref, hyp))

cfg = DERConfig(collar_s=0.25, skip_overlap=False)
result = corpus_der(pairs, cfg)
ci = der_bootstrap_ci(pairs, cfg, seed=0)
report = render_der_report([(OUR_SYSTEM, result, ci)], cfg, generated=ctx.timestamp)
(REPO / 'reports' / 'diarization.md').write_text(report, encoding='utf-8')
print(report)

## M6 — code-switch: our model on the CS slices → report

Same per-segment transcription; the report scores the code-switch slices under the
script-normalized policy automatically (slice domain = `code-switch`).

In [ ]:
from shono.transcribe import ShortFormWhisperTranscriber

our_short = ShortFormWhisperTranscriber(FT_MODEL, language='bn')
for slice_name in ['mucs-cs-test']:
    m = Manifest.from_jsonl(DATA / f'{slice_name}.manifest.jsonl')
    p = run_over_manifest(our_short, m, OUR_SYSTEM, audio_root=AUDIO_ROOT, run_context=ctx)
    p.to_jsonl(PREDS / f'{OUR_SLUG}__{slice_name}.jsonl')
!cd /kaggle/working/shono && python -m shono.eval --report codeswitch

## M7 — commercial APIs (within free credits) → full report

`BudgetGuard` refuses any run that would exceed the free allowance unless paid spend
is explicitly opted in — the $0 hard budget. Confirm current free allowances before
running; paid overage is a decision, not a default.

In [ ]:
from shono.api import BudgetGuard, DeepgramTranscriber, GoogleChirpTranscriber

EVAL_SLICES = ['cv-bn-test', 'bengali-loop-test', 'mucs-cs-test']
APIS = {
    'google-chirp': ('Google Chirp (bn-BD)',
                     GoogleChirpTranscriber(project_id=os.environ.get('GCP_PROJECT', ''))),
    'deepgram-nova3': ('Deepgram Nova-3 (bn)',
                       DeepgramTranscriber(api_key=os.environ.get('DEEPGRAM_KEY', ''))),
}
budget = BudgetGuard(free_hours=1.0, allow_paid=False)  # <-- confirm free allowance per API

for slug, (system_name, api) in APIS.items():
    for slice_name in EVAL_SLICES:
        m = Manifest.from_jsonl(DATA / f'{slice_name}.manifest.jsonl')
        p = run_over_manifest(api, m, system_name, audio_root=AUDIO_ROOT, budget=budget)
        p.to_jsonl(PREDS / f'{slug}__{slice_name}.jsonl')
        print(f'{slug} x {slice_name}: {len(p.hypotheses)} predictions')
!cd /kaggle/working/shono && python -m shono.eval --report full